In [ ]:

# SMART RESUME ANALYZER 
# Candidate Side System


import re
import PyPDF2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity



# TEXT CLEANING

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9 ]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()



# PDF TEXT EXTRACTION

def extract_text_from_pdf(pdf_path):
    text = ""
    try:
        with open(pdf_path, 'rb') as f:
            reader = PyPDF2.PdfReader(f)
            for page in reader.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text + " "
        return text
    except:
        print("Error reading PDF. Check file name.")
        return ""



# SKILL GAP ANALYSIS

def skill_gap_analysis(resume_text, job_text):
    resume_words = set(resume_text.split())
    job_words = set(job_text.split())
    missing = job_words - resume_words
    return list(missing)[:10]



# DOMAIN PREDICTION (RULE BASED CLASSIFIER)

def predict_domain(resume_text):

    domain_keywords = {
        "Data Science": ["python", "pandas", "numpy", "statistics", "analysis", "visualization", "power bi"],
        "AI / ML": ["machine learning", "deep learning", "cnn", "nlp", "tensorflow", "keras", "model", "scikit"],
        "Full Stack": ["html", "css", "javascript", "react", "node", "django", "flask", "api", "mongodb"]
    }

    scores = {}

    for domain, keywords in domain_keywords.items():
        score = 0
        for word in keywords:
            if word in resume_text:
                score += 1
        scores[domain] = score

    predicted_domain = max(scores, key=scores.get)

    return predicted_domain, scores



# MAIN SYSTEM FUNCTION

def resume_system():

    print("\nLoading Job Dataset...")
    jobs_df = pd.read_csv("jobs_dataset_with_features.csv")
    jobs_df.columns = jobs_df.columns.str.strip()
    jobs_df['Features'] = jobs_df['Features'].fillna('')

    print("\nChoose Resume Input Method:")
    print("1 - Upload Resume PDF")
    print("2 - Enter Resume Text")
    print("0 - Exit")

    choice = input("Enter 1, 2 or 0: ")

    if choice == "0":
        print("Exiting System... ")
        return False

    if choice == "1":
        pdf_path = input("Enter Resume PDF File Name: ")
        resume_text = extract_text_from_pdf(pdf_path)

    elif choice == "2":
        resume_text = input("Paste your resume text here:\n")

    else:
        print("Invalid choice!")
        return True

    if resume_text == "":
        print("Resume empty!")
        return True

    resume_clean = clean_text(resume_text)

    
    # TF-IDF SIMILARITY
    
    documents = [resume_clean] + list(jobs_df['Features'].apply(clean_text))

    vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
    tfidf_matrix = vectorizer.fit_transform(documents)

    similarity_scores = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:])
    scores = similarity_scores[0] * 100

    jobs_df['Similarity'] = scores.round(2)

    jobs_unique = jobs_df.sort_values(by="Similarity", ascending=False)
    jobs_unique = jobs_unique.drop_duplicates(subset=["Role"])

    top_jobs = jobs_unique.head(3)

    overall_score = round(np.mean(jobs_unique.head(5)['Similarity']), 2)

    
    # PRINT RESULTS
    
    print("\n==============================")
    print(f" Overall Resume Score: {overall_score} / 100")
    print("==============================\n")

    if overall_score >= 75:
        print("Resume Strength: Strong Profile ")
    elif overall_score >= 50:
        print("Resume Strength: Moderate Profile ")
    else:
        print("Resume Strength: Needs Improvement ")

    print("\n=====  Top 3 Career Recommendations =====\n")
    print(top_jobs[['Role', 'Similarity']])

    
    # DOMAIN PREDICTION
    
    domain, domain_scores = predict_domain(resume_clean)

    print("\n=====  Resume Domain Prediction =====")
    print("Predicted Domain:", domain)
    print("Domain Keyword Scores:", domain_scores)

    
    # SKILL GAP
    
    best_job_text = clean_text(top_jobs.iloc[0]['Features'])
    gaps = skill_gap_analysis(resume_clean, best_job_text)

    print("\n=====  Skill Gap (For Top Role) =====")
    print(gaps)

    
    # VISUALIZATION 1: Top 3 Jobs
    
    plt.figure()
    plt.bar(top_jobs['Role'], top_jobs['Similarity'])
    plt.xlabel("Job Role")
    plt.ylabel("Similarity Score")
    plt.title("Top 3 Job Match Scores")
    plt.xticks(rotation=45)
    plt.show()

    
    # VISUALIZATION 2: Domain Classification
    
    plt.figure()
    plt.bar(domain_scores.keys(), domain_scores.values())
    plt.xlabel("Domain")
    plt.ylabel("Keyword Match Count")
    plt.title("Resume Domain Classification")
    plt.show()

    
    # VISUALIZATION 3: Strength Pie Chart
    
    remaining = 100 - overall_score

    plt.figure()
    plt.pie([overall_score, remaining],
            labels=["Matched %", "Improvement Area %"],
            autopct='%1.1f%%')
    plt.title("Resume Strength Overview")
    plt.show()

    return True



# RUN CONTINUOUSLY


if __name__ == "__main__":

    while True:
        continue_program = resume_system()

        if continue_program == False:
            break

        print("\n-----------------------------------")
        print("System Ready for Next Resume")
        print("-----------------------------------")


Loading Job Dataset...

Choose Resume Input Method:
1 - Upload Resume PDF
2 - Enter Resume Text
0 - Exit


Enter 1, 2 or 0:  1
